This notebook applies ClusterDE to find marker genes between adjacent genes in the pancreas endocrinogenesis data. The data fall along a continuous pseudotime trajectory, and there are no sharp cluster boundaries. That said, there can be true mean differences across the cell types, since the clustering along the pseudotime trajectory isn't completely arbitrary. Our initial p-values are computed using Wilcoxon test, the correction uses Clipper with the Data Splitting-based threshold.

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
from scdesigner.datasets import pancreas
from clusterde_py import find_markers

GROUP1 = "Ngn3 low EP"
GROUP2 = "Ngn3 high EP"
FDR = 0.05
SEED = 0

First we run a standard marker gene analysis without any ClusterDE synthetic null control.

In [ ]:
adata = pancreas()
adata_filered = adata
adata_filered = adata[adata.obs["cell_type"].isin([GROUP1, GROUP2])].copy()
adata_filered.obs["cell_type"] = adata_filered.obs["cell_type"].cat.remove_unused_categories()

# Naive DE: log-normalize + one-sided Wilcoxon
naive = adata_filered.copy()
sc.pp.normalize_total(naive)
sc.pp.log1p(naive)
sc.tl.rank_genes_groups(
    naive, "cell_type", groups=[GROUP1], reference=GROUP2, method="wilcoxon"
)
naive_df = sc.get.rank_genes_groups_df(naive, GROUP1)
naive_df = naive_df[naive_df["logfoldchanges"] > 0]
naive_significant = naive_df.loc[naive_df["pvals_adj"] <= FDR, "names"]


Next we run ClusterDE.

In [ ]:
result = find_markers(
    adata_filered,
    "cell_type",
    GROUP1,
    GROUP2,
    fdr=FDR,
    null_kwargs={"seed": SEED},
    cluster_kwargs={"seed": SEED},
)
clusterde_significant = result[result["is_DE"]]

These genes were found significant in the naive marker gene analysis.

In [ ]:
naive_significant

After running clusterde, only a handful remain significant.

In [ ]:
clusterde_significant

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(result["contrast_score"], bins=20, color="black")
ax.axvline(0, color="#C73333", linewidth=1)
ax.set_xlabel("Contrast score (target - null, -log10 p)")
ax.set_ylabel("Number of genes")
ax.set_title(f"ClusterDE contrast scores")
fig.tight_layout()
fig.show()